# Diseño de los resonadores de readout para el diseño

Objetivo: obtener y caracterizar los parámetros geométricos necesarios para el acoplamiento de los resonadores a cada qubit transmon y a la línea de transmisión.

Parámetros a caracterizar:

- [ ] Capacitancia de acoplamiento al transmon (y magnitud del acoplamiento) en función de las dimensiones del transmon y del coupler
- [ ] Ancho de línea y factor de calidad de los resonadores en función de la separación a la línea de transmisión y la longitud de acoplamiento
- [ ] Velocidad de la luz e impedancia de los resonadores 

In [2]:
from collections import OrderedDict

from qiskit_metal import designs, Dict
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal import MetalGUI

import numpy as np
from scipy.optimize import brentq
from parametric_coupling.design_helper import DesignHelper

In [3]:
import os
import shapely
from shapely.ops import unary_union


def component_trace_union(design, names):
    """Union of the trace (non-subtract) polys/paths belonging to the given
    component names, as real shapely geometry -- used for overlap checks."""
    ids = [design.components[n].id for n in names if n in design.components]
    polys = []
    pt, pp = design.qgeometry.tables["path"], design.qgeometry.tables["poly"]
    m = (~pt["subtract"].astype(bool)) & pt["component"].isin(ids)
    for _, r in pt[m].iterrows():
        w = float(r.get("width") or 0.0) or 0.01
        polys.append(r["geometry"].buffer(w / 2.0, cap_style=2, join_style=2))
    m = (~pp["subtract"].astype(bool)) & pp["component"].isin(ids)
    polys += list(pp[m]["geometry"])
    return unary_union(polys) if polys else shapely.geometry.GeometryCollection()


def resonator_overlap_um2(design, other_names=("TL1", "TL2", "CLT", "LP1", "LP2")):
    """resonator<->(other_names) overlap area (um^2) -- should always be 0.
    Includes the launchpads (not just the feedline/coupler): a wide or tall
    snake can just as easily run into a launchpad taper as into the TL
    itself. A NUMERIC check, deliberately independent of any visual render --
    for the visual, use gui.screenshot() (the real qiskit_metal renderer),
    not a custom matplotlib approximation: a custom render based on naively
    buffering the 'path'/'poly' tables was tried in an earlier version of
    this notebook and did not faithfully reproduce fillets, which is exactly
    where this design's real geometry problems (crowded/self-intersecting
    turns) showed up -- the qiskit_metal GUI's own renderer is the only one
    that actually shows those correctly. Call this right after every rebuild
    -- both in the sweep below and (see build_resonator) as a self-check
    inside the build function itself, so a bad geometry fails immediately
    with a clear message rather than silently producing a disfigured
    resonator."""
    res = component_trace_union(design, ["readout_1"])
    other = component_trace_union(design, list(other_names))
    if res.is_empty or other.is_empty:
        return 0.0
    return res.intersection(other).area * 1e6

In [4]:
design = designs.DesignPlanar({}, overwrite_enabled=True)
design.chips.main.size.size_x = '4.0mm'
design.chips.main.size.size_y = '4.0mm'
design.chips.main.size.size_z = '500um' # tamaño del sustrato y de la capa de aire a simular
design.chips.main.size.center_x = '0mm'
design.chips.main.size.center_y = '0.0mm'

design.variables['cpw_width'] = '20 um'
design.variables['cpw_gap'] = '11.5um'  # ~50 Ohm at cpw_width=20um, see the DesignHelper check below

## Verificación de impedancia de 50 Ω

Antes de construir la línea de transmisión, confirmamos con la fórmula
analítica de `DesignHelper.compute_cpw_geometric_parameters` (Ramos et al.,
[arXiv:2508.03873v1](https://arxiv.org/html/2508.03873v1), Apéndice B) que la
geometría CPW elegida arriba (`cpw_width=20um`, `cpw_gap=11.5um`, elegido para
dar Z0 ≈ 50 Ω -- ver el cálculo abajo) da Z0 ≈ 50 Ω, y de paso calculamos la
longitud λ/4 que le corresponde a un resonador de 8 GHz con esa misma sección
transversal (el valor que reutilizamos más abajo para `TARGET_RES_LENGTH_MM`).

**Nota:** `DesignHelper.compute_resonator_length`/`compute_resonator_frequency`
usan la velocidad de la luz en el VACÍO corregida solo por `alpha_inductance`
(la fracción cinética) — a diferencia de `compute_cpw_geometric_parameters`,
NO incluyen el `eps_eff` del CPW (que para esta sección transversal es ≈6.2,
es decir v_fase ≈ c/2.5). Por eso acá la longitud del resonador se calcula
directamente con la `v` de `compute_cpw_geometric_parameters`, no con
`compute_resonator_length` — usar ese método tal cual daría una longitud
~2.5x demasiado larga.

In [5]:
dh = DesignHelper()

CPW_WIDTH_UM = 20.0
CPW_GAP_UM = 11.5              # must match design.variables['cpw_gap'] above -- chosen for ~50 Ohm
SUBSTRATE_THICKNESS_NM = 100.0  # representative thin-film thickness for the B6/B7 correction
EPS_R = 11.45                    # SQDMetal Material('silicon') -- matches every Palace sim in this repo

geo = dh.compute_cpw_geometric_parameters(
    width=CPW_WIDTH_UM, gap=CPW_GAP_UM, thickness=SUBSTRATE_THICKNESS_NM * 1e-3, eps_r=EPS_R)
Z0_current = geo['Z0'].to('ohm').magnitude
print(f"cpw_width={CPW_WIDTH_UM}um cpw_gap={CPW_GAP_UM}um -> "
      f"Z0={geo['Z0']:.3f}  v={geo['v'].to('m/s'):.4e}  eps_eff={geo['eps_eff']:.3f}")

def _z0_at_gap(gap_um):
    return dh.compute_cpw_geometric_parameters(
        width=CPW_WIDTH_UM, gap=gap_um, thickness=SUBSTRATE_THICKNESS_NM * 1e-3, eps_r=EPS_R
    )['Z0'].to('ohm').magnitude - 50.0

gap_50ohm_um = brentq(_z0_at_gap, 5.0, 30.0)
print(f"Z0 at the current gap is {100*(Z0_current/50.0 - 1):+.3f}% off 50 Ohm "
      f"({Z0_current:.3f} Ohm); the exact-50-Ohm gap at cpw_width={CPW_WIDTH_UM}um is "
      f"{gap_50ohm_um:.3f}um, so CPW_GAP_UM=11.5um above is a {100*(CPW_GAP_UM/gap_50ohm_um-1):+.2f}% "
      f"rounding of it.")

# lambda/4 length for an 8 GHz fundamental at this cross-section, from the ACTUAL
# CPW phase velocity (see the note above on why NOT compute_resonator_length)
TARGET_FREQ_GHZ = 8.0
v_phase = geo['v'].to('m/s').magnitude
TARGET_RES_LENGTH_MM = v_phase / (4 * TARGET_FREQ_GHZ * 1e9) * 1e3
print(f"lambda/4 length @ {TARGET_FREQ_GHZ} GHz: {TARGET_RES_LENGTH_MM:.5f} mm  "
      f"(squid_array_coupling/build_design.py's RES_LENGTH_MM=3.74575 was tuned at "
      f"cpw_gap=12.25um, a different cross-section -- not directly comparable here)")

cpw_width=20.0um cpw_gap=11.5um -> Z0=49.961 ohm  v=1.2072e+08 meter / second  eps_eff=6.167
Z0 at the current gap is -0.079% off 50 Ohm (49.961 Ohm); the exact-50-Ohm gap at cpw_width=20.0um is 11.531um, so CPW_GAP_UM=11.5um above is a -0.27% rounding of it.
lambda/4 length @ 8.0 GHz: 3.77259 mm  (squid_array_coupling/build_design.py's RES_LENGTH_MM=3.74575 was tuned at cpw_gap=12.25um, a different cross-section -- not directly comparable here)


In [6]:
gui = MetalGUI(design)
gui.rebuild()
gui.autoscale()

In [23]:
try:
    LP1.delete(); LP2.delete(); CLT.delete(); TL1.delete(); TL2.delete()
except NameError:
    pass

fillet_radius_um = 75

def build_feedline_and_coupler(coupling_space_um, coupling_length_um, down_length_um=150.0,
                               chip_half_span_um=5000.0):
    """Feedline (LP1 -- TL1 -- CoupledLineTee -- TL2 -- LP2), with the
    resonator's coupled ("hanger") end built into the CoupledLineTee itself.

    ``coupling_space_um`` is the separation to the transmission line and
    ``coupling_length_um`` is the colinear coupled length -- the two
    parameters to sweep for the coupling quality factor Qc.
    ``down_length_um`` is normally DERIVED from a fixed budget (see below), not
    passed independently -- see build_resonator's docstring for why.
    """
    assert coupling_length_um < 0.8 * chip_half_span_um, (
        f"coupling_length_um={coupling_length_um} too large for the "
        f"{chip_half_span_um}um launchpad-to-center span")
    assert down_length_um > 0, (
        f"down_length_um={down_length_um} <= 0 (coupling_length_um={coupling_length_um} "
        f"exceeds COUPLING_BUDGET_UM) -- CoupledLineTee's own 'down' segment then runs "
        f"BACKWARDS (toward the feedline instead of away from it), which is exactly what "
        f"produced the resonator-overlaps-the-TL bug once. Raise COUPLING_BUDGET_UM.")

    x1, y1 = f'-{chip_half_span_um}um', '0um'
    x2 = f'{chip_half_span_um}um'

    launch_options1 = dict(chip='main', pos_x=x1, pos_y=y1, orientation='360',
                           lead_length='30um', pad_height='400um', pad_width='250um',
                           pad_gap='150um', taper_height='350um')
    lp1 = LaunchpadWirebond(design, 'LP1', options=launch_options1)

    launch_options2 = dict(chip='main', pos_x=x2, pos_y=y1, orientation='180',
                           lead_length='30um', pad_height='400um', pad_width='250um',
                           pad_gap='150um', taper_height='350um')
    lp2 = LaunchpadWirebond(design, 'LP2', options=launch_options2)

    # prime_* / second_* match the main cpw_width/cpw_gap -- the resonator's own
    # cross-section is the same 50-Ohm-ish line verified above, not a separate one.
    clt = CoupledLineTee(design, 'CLT', options=dict(
        pos_x='0um', pos_y='0um', orientation='0', chip='main',
        prime_width='cpw_width', prime_gap='cpw_gap',
        second_width='cpw_width', second_gap='cpw_gap',
        coupling_space=f'{coupling_space_um}um', coupling_length=f'{coupling_length_um}um',
        down_length=f'{down_length_um}um', fillet=f'{fillet_radius_um}um', mirror=False,
        open_termination=True))

    tl1 = RoutePathfinder(design, 'TL1', options=dict(
        chip='main', trace_width='cpw_width', trace_gap='cpw_gap', fillet=f'{fillet_radius_um}um',
        hfss_wire_bonds=True, lead=dict(end_straight='0.1mm'),
        pin_inputs=Dict(start_pin=Dict(component='LP1', pin='tie'),
                        end_pin=Dict(component='CLT', pin='prime_start'))))
    tl2 = RoutePathfinder(design, 'TL2', options=dict(
        chip='main', trace_width='cpw_width', trace_gap='cpw_gap', fillet=f'{fillet_radius_um}um',
        hfss_wire_bonds=True, lead=dict(start_straight='0.1mm'),
        pin_inputs=Dict(start_pin=Dict(component='CLT', pin='prime_end'),
                        end_pin=Dict(component='LP2', pin='tie'))))
    return lp1, lp2, clt, tl1, tl2


# --- sweep parameters: separation to the TL and colinear coupled length ---
COUPLING_SPACE_UM = 15.0     # "separación a la línea de transmisión"
COUPLING_LENGTH_UM = 200.0   # "longitud de acoplamiento" (colinear length)

# coupling_length_um + down_length_um is held at this fixed budget, so
# down_length_um below is DERIVED, not chosen independently -- this keeps
# res_short's position RELATIVE to CLT's second_end pin identical no matter
# what coupling_length_um/coupling_space_um are (see build_resonator's
# docstring), which keeps the meander's shape identical across the whole
# sweep too: only the coupled region changes, not the resonator's overall
# geometry -- nothing "drags" the meander body around between sweep points.
#
# MUST stay comfortably above the largest coupling_length_um you plan to
# sweep (leaving down_length_um >= ~100um of routing clearance) -- 550 covers
# the example sweep below (max coupling_length_um=400) with 150um to spare.
# Verified (rendered + checked for overlap) across that whole grid.
COUPLING_BUDGET_UM = 500.0
DOWN_LENGTH_UM = COUPLING_BUDGET_UM - COUPLING_LENGTH_UM

LP1, LP2, CLT, TL1, TL2 = build_feedline_and_coupler(
    COUPLING_SPACE_UM, COUPLING_LENGTH_UM, DOWN_LENGTH_UM)
design.rebuild()
gui.rebuild()
gui.autoscale()

11:06PM 56s ERROR [rebuild]: ERROR in building component name=readout_1, error='NoneType' object has no attribute 'coords'


AttributeError: 'NoneType' object has no attribute 'coords'

## Cuerpo del resonador (λ/4)

El extremo acoplado del resonador ya queda definido dentro del
`CoupledLineTee` de arriba (el pin `second_end`; el extremo cercano a la línea
queda eléctricamente abierto vía `open_termination=True`). Desde ahí, el
resonador es un camino tipo "víbora" (boustrophedon), calculado punto por
punto en Python y dibujado con `RouteAnchors` (waypoints exactos, sin ajuste
automático de longitud -- ver por qué en el docstring de
`build_snake_anchors`):

  - un tramo recto corto hacia abajo,
  - gira y va hacia el Este hasta `x = +x_max_um` (fila 1),
  - gira, baja `row_step_um` (transición),
  - gira, va hacia el Oeste hasta `x = -x_max_um` (fila 2),
  - gira, baja `row_step_um`, gira, fila 3 hacia el Este, ... y así hasta
    agotar la longitud que falta, terminando en un `ShortToGround` real
    ubicado exactamente donde termina la víbora, con la orientación que
    corresponde a la dirección en la que venía viajando.

`x_max_um` (950um por default) mantiene la víbora lejos de los launchpads
(a ±1000um) sin costar ancho de chip extra -- ese ancho ya está "gastado" por
la línea de transmisión. Cada fila adicional sólo cuesta ALTURA de chip
(`row_step_um`), que es la dimensión más barata de agregar. Esto es lo
opuesto al primer diseño (una columna vertical de pliegues horizontales
angostos), que usaba altura libremente pero se acercaba a los launchpads a
medida que crecía `COUPLING_LENGTH_UM`.

La longitud total (acoplador + víbora) se mantiene fija en
`TARGET_RES_LENGTH_MM` (calculada arriba para 8 GHz) sin importar cómo
cambien `COUPLING_SPACE_UM`/`COUPLING_LENGTH_UM` -- así el barrido cambia el
acoplamiento (y por lo tanto Qc) sin correr la frecuencia de resonancia.
`build_resonator` verifica con `resonator_overlap_um2` que la víbora
construida no se solapa con nada antes de devolver el componente, así que un
resonador deformado nunca queda armado en silencio.

In [21]:
try:
    res_short.delete(); readout_1.delete()
except NameError:
    pass


def build_snake_anchors(start_xy_mm, remaining_mm, x_max_um=950.0, row_step_um=200.0,
                        initial_lead_um=100.0, min_seg_um=100.0):
    """Boustrophedon waypoints (um, absolute) starting from ``start_xy_mm``
    (= CLT's 'second_end' pin, heading down):

        straight down initial_lead_um
        -> turn, go EAST to x = start_x + x_max_um            (row 1)
        -> turn, go DOWN row_step_um                          (transition)
        -> turn, go WEST to x = start_x - x_max_um            (row 2)
        -> turn, go DOWN row_step_um                          (transition)
        -> turn, go EAST to x = start_x + x_max_um            (row 3)
        -> ... repeat until remaining_mm is used up.

    This is tracked directly in Python (position + accumulated length), not
    via qiskit_metal's own turn/jog DSL or its automatic meander-length
    fitting -- both were tried first and both fought this exact shape (see
    git history of this cell): RouteMeander's ``connect_meandered`` picks its
    OWN fold count/depth from the direct pin-to-pin distance regardless of
    jogs, and mixing ``start_jogged_extension`` with a ``total_length`` target
    that the jogs already fully account for left ``_length_segment`` negative
    and produced dozens of degenerate micro-segments. Tracking the endpoint
    explicitly and handing qiskit_metal exact waypoints (via ``RouteAnchors``,
    no length-fitting involved at all) sidesteps that entirely.

    A final leg shorter than ``min_seg_um`` is clamped UP to it rather than
    left too short for the fillet to round cleanly (a few tens of um of
    overshoot out of several mm is a negligible frequency error).

    Returns ``(waypoints_um, final_heading)``, ``waypoints_um`` a list of
    ``(x_um, y_um)`` INCLUDING the very first point (right after the initial
    lead) through the LAST point (where res_short belongs -- not included as
    an anchor, since it is a real pin, not a waypoint); ``final_heading`` in
    ``{'EAST', 'WEST', 'DOWN'}``, needed to orient res_short correctly.
    """
    x0, y0 = start_xy_mm[0] * 1e3, start_xy_mm[1] * 1e3
    remaining_um = remaining_mm * 1e3 - initial_lead_um
    assert remaining_um > 0, "remaining_mm too short for even the initial lead"

    x, y = x0, y0 - initial_lead_um
    pts = [(x, y)]

    seg = x_max_um if remaining_um >= x_max_um else max(remaining_um, min_seg_um)
    x += seg; remaining_um -= seg
    pts.append((x, y))
    row_dir, heading = +1, 'EAST'

    while remaining_um > 1e-6:
        seg = row_step_um if remaining_um >= row_step_um else max(remaining_um, min_seg_um)
        y -= seg; remaining_um -= seg
        pts.append((x, y))
        heading = 'DOWN'
        if remaining_um <= 1e-6:
            break
        row_dir = -row_dir
        full_seg = 2 * x_max_um   # full row width (start_x +- x_max_um each side)
        seg = full_seg if remaining_um >= full_seg else max(remaining_um, min_seg_um)
        x += row_dir * seg; remaining_um -= seg
        pts.append((x, y))
        heading = 'EAST' if row_dir > 0 else 'WEST'

    return pts, heading


def build_resonator(coupling_length_um, down_length_um, target_length_mm=TARGET_RES_LENGTH_MM,
                    x_max_um=950.0, row_step_um=200.0, initial_lead_um=100.0, fillet_um=40.0):
    """RouteAnchors snake (see build_snake_anchors) from CLT's 'second_end' pin
    to a ShortToGround placed at the snake's own computed endpoint, oriented
    to match the direction the snake was heading when it stopped.

    total_length (coupler + snake) is fixed at target_length_mm regardless of
    coupling_length_um/down_length_um -- keeps f_res fixed while
    coupling_space_um/coupling_length_um are swept for Qc. ``x_max_um`` keeps
    the snake within +-x_max_um of the coupler's own x position: 950um stays
    clear of the launchpad tapers (at +-1000um) with a 50um margin. Growing
    DOWN (more rows) costs nothing extra in chip width, only height -- the
    opposite trade of the very first (pure vertical column) version of this
    resonator, which used chip height freely but crept toward the launchpads
    sideways as coupling_length_um grew.
    """
    consumed_mm = (coupling_length_um + down_length_um) * 1e-3
    remaining_mm = target_length_mm - consumed_mm
    if remaining_mm <= 0.2:
        raise ValueError(f"remaining_mm={remaining_mm:.3f} too short to route "
                         f"(coupling_length_um + down_length_um already consumed "
                         f"{consumed_mm*1e3:.0f}um of the {target_length_mm:.3f}mm target)")

    se = design.components['CLT'].pins['second_end']['middle']   # mm, [x, y]
    pts_um, heading = build_snake_anchors(se, remaining_mm, x_max_um, row_step_um, initial_lead_um)

    orientation = {'EAST': '0', 'WEST': '180', 'DOWN': '270'}[heading]
    end_x_um, end_y_um = pts_um[-1]
    r_short = ShortToGround(design, 'res_short', options=dict(
        pos_x=f'{end_x_um}um', pos_y=f'{end_y_um}um', orientation=orientation,
        chip='main', width='cpw_width'))

    anchors = OrderedDict()
    for i, (x_um, y_um) in enumerate(pts_um[:-1]):    # last point is res_short's own pin, not an anchor
        anchors[i] = np.array([x_um, y_um]) * 1e-3    # mm

    r1 = RouteAnchors(design, 'readout_1', options=dict(
        pin_inputs=dict(
            start_pin=dict(component='CLT', pin='second_end'),
            end_pin=dict(component='res_short', pin='short'),
        ),
        fillet=f'{fillet_um}um',
        anchors=anchors,
        lead=dict(start_straight='2um', end_straight='2um'),
        trace_width='cpw_width', trace_gap='cpw_gap',
    ))

    design.rebuild()
    overlap = resonator_overlap_um2(design)
    if overlap > 1e-6:
        raise RuntimeError(f"build_resonator: readout_1 overlaps the coupler/TL/launchpads "
                           f"by {overlap:.2f}um^2 -- lower x_max_um (most likely fix) or "
                           f"row_step_um.")

    print(f"resonator: coupler+down consumed {consumed_mm:.3f}mm, snake target "
          f"{remaining_mm:.3f}mm (actual path length {r1.length*1e3:.3f}mm -- "
          f"a bit shorter than the straight-segment sum, fillets cut the corners), "
          f"final heading {heading}")
    return r_short, r1


res_short, readout_1 = build_resonator(COUPLING_LENGTH_UM, DOWN_LENGTH_UM)
design.rebuild()
gui.rebuild()
gui.autoscale()

09:27PM 57s INFO [delete_component]: Called delete_component res_short, but such a component is not in the design cache dictionary of components.
09:27PM 57s INFO [delete_component]: Called delete_component readout_1, but such a component is not in the design cache dictionary of components.


resonator: coupler+down consumed 0.500mm, snake target 3.273mm (actual path length 3203.920mm -- a bit shorter than the straight-segment sum, fillets cut the corners), final heading DOWN


In [15]:
CLT.options

{'pos_x': '0um',
 'pos_y': '0um',
 'orientation': '0',
 'chip': 'main',
 'layer': '1',
 'prime_width': 'cpw_width',
 'prime_gap': 'cpw_gap',
 'second_width': 'cpw_width',
 'second_gap': 'cpw_gap',
 'coupling_space': '15.0um',
 'coupling_length': '200.0um',
 'down_length': '150.0um',
 'fillet': '75um',
 'mirror': False,
 'open_termination': True,
 'hfss_wire_bonds': False,
 'q3d_wire_bonds': False,
 'aedt_q3d_wire_bonds': False,
 'aedt_hfss_wire_bonds': False}

## Cómo barrer separación y longitud de acoplamiento

Para el barrido de Qc: llamar de nuevo a `build_feedline_and_coupler(...)` +
`build_resonator(...)` con distintos `COUPLING_SPACE_UM`/`COUPLING_LENGTH_UM`,
derivando `down_length_um = COUPLING_BUDGET_UM - COUPLING_LENGTH_UM` en vez de
fijarlo aparte (`COUPLING_BUDGET_UM` sí queda fijo), y volver a
`design.rebuild()`. Mantener el presupuesto fijo es lo que mantiene el cuerpo
del resonador (la víbora) con la MISMA forma en todo el barrido -- ver el
docstring de `build_resonator`. Cada punto se simula por separado en el
**cluster**, nunca localmente (ver `directives/` y las convenciones del resto
del repo, e.g. `notebooks/design/cpw_impedance_test/`): una simulación de
capacitancia (o un driven/eigenmode sim) para extraer la capacitancia de
acoplamiento `Cc` en cada punto, que junto con
`DesignHelper.compute_resonator_Qc_from_Cc` da Qc analíticamente -- o, si el
sim mide S21/el ancho de línea directamente, Qc sale de ahí sin pasar por
`Cc`.

`resonator_overlap_um2` (definido justo después de los imports, arriba) es un
chequeo NUMÉRICO barato de que el resonador no se solapa con la
línea/acoplador/launchpads -- `build_resonator` ya lo llama internamente y
lanza un error si falla, así que un resonador deformado nunca llega a
construirse en silencio (esto pasó más de una vez durante el desarrollo).
Para inspección VISUAL usar `gui.screenshot(name=...)` -- el renderer real de
qiskit_metal -- no un render propio: una versión anterior de esta libreta
armaba las imágenes bufferizando a mano las tablas `path`/`poly`, y esa
aproximación no reproducía bien los fillets, justo donde aparecían los
problemas reales de geometría (curvas apretadas/auto-intersectadas).

La celda siguiente confirma, LOCALMENTE y sin ninguna simulación, que la
grilla completa de geometrías construye sin errores NI overlap, y guarda una
captura (`gui.screenshot`) por punto en `sweep_screenshots/` -- el paso antes
de exportar esto a un `build_design.py` para el barrido real en el cluster
(mismo patrón que `notebooks/design/cpw_impedance_test/build_design.py`).

In [22]:
# Geometry-only sanity sweep (NO simulation) -- confirms every grid point
# builds without error AND without the resonator overlapping the TL/launchpads
# (build_resonator already self-checks this and raises immediately if not, so
# reaching the "OK" print below is itself a real confirmation, not just a
# hope), and saves a screenshot of each point to sweep_screenshots/ via
# gui.screenshot() -- the real qiskit_metal renderer (see resonator_overlap_um2's
# docstring for why not a custom one). This is the step before exporting this
# into a build_design.py for the cluster.
sweep_points = [
    dict(coupling_space_um=cs, coupling_length_um=cl)
    for cs in (10.0, 15.0, 25.0)
    for cl in (150.0, 250.0, 400.0)
]

os.makedirs("sweep_screenshots", exist_ok=True)

for i, pt in enumerate(sweep_points):
    try:
        LP1.delete(); LP2.delete(); CLT.delete(); TL1.delete(); TL2.delete()
        res_short.delete(); readout_1.delete()
    except NameError:
        pass
    # down_length_um DERIVED from the same fixed budget as the driver cell
    # above -- keeps the resonator's shape identical across every sweep point
    # (see build_resonator's docstring).
    down_length_um = COUPLING_BUDGET_UM - pt['coupling_length_um']
    LP1, LP2, CLT, TL1, TL2 = build_feedline_and_coupler(
        pt['coupling_space_um'], pt['coupling_length_um'], down_length_um)
    res_short, readout_1 = build_resonator(pt['coupling_length_um'], down_length_um)
    design.rebuild()

    overlap = resonator_overlap_um2(design)
    print(f"OK: coupling_space={pt['coupling_space_um']}um "
          f"coupling_length={pt['coupling_length_um']}um  overlap={overlap:.4f}um^2")

    gui.rebuild()
    gui.autoscale()
    fname = f"pt{i}_cs{pt['coupling_space_um']:g}_cl{pt['coupling_length_um']:g}"
    gui.screenshot(name=f"sweep_screenshots/{fname}", display=False)

# leave the GUI showing the LAST point of the sweep
gui.rebuild()
gui.autoscale()

resonator: coupler+down consumed 0.500mm, snake target 3.273mm (actual path length 3203.920mm -- a bit shorter than the straight-segment sum, fillets cut the corners), final heading DOWN
OK: coupling_space=10.0um coupling_length=150.0um  overlap=0.0000um^2
resonator: coupler+down consumed 0.500mm, snake target 3.273mm (actual path length 3203.920mm -- a bit shorter than the straight-segment sum, fillets cut the corners), final heading DOWN
OK: coupling_space=10.0um coupling_length=250.0um  overlap=0.0000um^2
resonator: coupler+down consumed 0.500mm, snake target 3.273mm (actual path length 3203.920mm -- a bit shorter than the straight-segment sum, fillets cut the corners), final heading DOWN
OK: coupling_space=10.0um coupling_length=400.0um  overlap=0.0000um^2
resonator: coupler+down consumed 0.500mm, snake target 3.273mm (actual path length 3203.920mm -- a bit shorter than the straight-segment sum, fillets cut the corners), final heading DOWN
OK: coupling_space=15.0um coupling_length=